# 0. Setup & Installation
Run this cell first. Restart runtime if prompted.

In [4]:
!pip install -U transformers bitsandbytes accelerate datasets scikit-learn --quiet

# 1. Authentication

In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_1)

# 2. Configuration

In [6]:
from dataclasses import dataclass


@dataclass
class ExperimentConfig:
    # ---- Model ----
    model_name: str = "Qwen/Qwen3-4B"
    max_seq_length: int = 2048

    # ---- Quantization ----
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True

    # ---- Generation ----
    max_new_tokens: int = 5


cfg = ExperimentConfig()
print(f"Config: {cfg}")

Config: ExperimentConfig(model_name='Qwen/Qwen3-4B', max_seq_length=2048, load_in_4bit=True, bnb_4bit_quant_type='nf4', use_double_quant=True, max_new_tokens=5)


# 3. Load Eval Data

In [7]:
import pandas as pd

eval_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/prepare-test-data-0327/test.csv")

print(f"Eval: {len(eval_data)} samples")
eval_data.head()

Eval: 200 samples


,conversation_id,conversation,full_text,outcome,text,transcript
0,saas-12-conv-4816,"[{""speaker"": ""customer"", ""message"": ""Hey, I sa...","Hey, I saw your post about SmartLLM's integrat...",1,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, I saw your ..."
1,saas-1-conv-4852,"[{""speaker"": ""customer"", ""message"": ""Hey, so, ...","Hey, so, I've been looking into some pricing t...",0,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, so, I've be..."
2,saas-0-conv-1824,"[{""speaker"": ""customer"", ""message"": ""Hey, than...","Hey, thanks for hopping on this call! So, um, ...",1,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, thanks for ..."
3,saas-10-conv-3173,"[{""speaker"": ""sales_rep"", ""message"": ""Hey Jess...","Hey Jessica, hope you're doing well! I wanted ...",0,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|agent|>Hey Jessica, hope y..."
4,saas-18-conv-466,"[{""speaker"": ""customer"", ""message"": ""Hey, so I...","Hey, so I've been, like, struggling with track...",0,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, so I've bee..."


# 4. Load Base Model + Tokenizer (No LoRA)

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- Quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

# ---- Load tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer vocab size: {len(tokenizer)}")

# ---- Load base model (no LoRA) ----
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
)
model.eval()

print(f"Base model loaded: {cfg.model_name}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base model loaded: Qwen/Qwen3-4B


# 5. Inference Functions

Ask the base model to directly output a conversion probability (0 to 1). Parse the first number from the output; fall back to a random value if unparseable.

In [9]:
import re
import random

SYSTEM_PROMPT = (
    "You are an expert sales call analyst specializing in financial services. "
    "You will be given a full transcript of a customer-agent conversation. "
    "Predict the probability that the customer will convert, from 0 to 1. "
    "Output a number and nothing else."
)


def build_inference_prompt(system_prompt: str, transcript: str) -> str:
    """Build a chat-style prompt for the base Qwen3 model."""
    return (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{transcript}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def parse_probability(text: str) -> float:
    """Extract the first number from generated text. Random fallback."""
    match = re.match(r"^\s*([01]?\.?\d*)", text)
    if match and match.group(1):
        try:
            val = float(match.group(1))
            return max(0.0, min(1.0, val))
        except ValueError:
            pass
    return random.random()


def predict(model, tokenizer, transcript: str, max_new_tokens=5):
    prompt = build_inference_prompt(SYSTEM_PROMPT, transcript)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    prob = parse_probability(new_tokens)
    return {
        "generated_output": new_tokens,
        "conversion_probability": prob,
        "predicted_class": 1 if prob > 0.5 else 0,
    }

# 6. Test Inference on One Example

In [10]:
test_row = eval_data.iloc[0]
test_transcript = test_row["transcript"]
test_label = test_row["outcome"]

result = predict(model, tokenizer, test_transcript, max_new_tokens=cfg.max_new_tokens)

print(f"Ground truth: {test_label}")
print(f"Generated output: '{result['generated_output']}'")
print(f"Parsed probability: {result['conversion_probability']:.4f}")
print(f"Predicted class: {result['predicted_class']}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Ground truth: 1
Generated output: '<think>
Okay, let'
Parsed probability: 0.3790
Predicted class: 0


# 7. Evaluate on Full Eval Set

In [11]:
from sklearn.metrics import roc_auc_score, classification_report

eval_results = []
fallback_count = 0

print("Running evaluation...")
for i in range(len(eval_data)):
    row = eval_data.iloc[i]
    result = predict(model, tokenizer, row["transcript"], max_new_tokens=cfg.max_new_tokens)

    # Track if fallback was used (output didn't start with a number)
    raw = result["generated_output"]
    if not re.match(r"^\s*[01]?\.?\d", raw):
        fallback_count += 1

    eval_results.append({
        "ground_truth": row["outcome"],
        "predicted_class": result["predicted_class"],
        "probability": result["conversion_probability"],
        "generated_output": raw,
    })
    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(eval_data)} samples")

# ---- Compute metrics ----
y_true = [r["ground_truth"] for r in eval_results]
y_pred = [r["predicted_class"] for r in eval_results]
y_prob = [r["probability"] for r in eval_results]

correct = sum(1 for r in eval_results if r["ground_truth"] == r["predicted_class"])
total = len(eval_results)
accuracy = correct / total

print(f"\n{'='*60}")
print(f"EVALUATION RESULTS (Base Qwen3-4B, no LoRA)")
print(f"{'='*60}")
print(f"Accuracy: {correct}/{total} = {accuracy:.2%}")
print(f"AUC-ROC: {roc_auc_score(y_true, y_prob):.4f}")
print(f"Fallback (random) predictions: {fallback_count}/{total}")
print(f"\n{classification_report(y_true, y_pred, target_names=['No Convert (0)', 'Convert (1)'])}")

Running evaluation...
  Processed 10/200 samples
  Processed 20/200 samples
  Processed 30/200 samples
  Processed 40/200 samples
  Processed 50/200 samples
  Processed 60/200 samples
  Processed 70/200 samples
  Processed 80/200 samples
  Processed 90/200 samples
  Processed 100/200 samples
  Processed 110/200 samples
  Processed 120/200 samples
  Processed 130/200 samples
  Processed 140/200 samples
  Processed 150/200 samples
  Processed 160/200 samples
  Processed 170/200 samples
  Processed 180/200 samples
  Processed 190/200 samples
  Processed 200/200 samples

EVALUATION RESULTS (Base Qwen3-4B, no LoRA)
Accuracy: 93/200 = 46.50%
AUC-ROC: 0.4638
Fallback (random) predictions: 200/200

                precision    recall  f1-score   support

No Convert (0)       0.47      0.51      0.49       100
   Convert (1)       0.46      0.42      0.44       100

      accuracy                           0.47       200
     macro avg       0.46      0.46      0.46       200
  weighted avg    

# 8. Cross-Entropy with Ground Truth (Dirac Delta)

For a Dirac delta ground-truth distribution where $q(y=k) = 1$ for the true class and $0$ otherwise, the cross-entropy simplifies to:

$$H(q, p) = -\log p(y = y_{\text{true}})$$

where $p(y=1)$ is the model's predicted conversion probability.

In [12]:
import numpy as np

y_true_arr = np.array(y_true)
y_prob_arr = np.array(y_prob)

# Clip probabilities to avoid log(0)
eps = 1e-7
y_prob_clipped = np.clip(y_prob_arr, eps, 1.0 - eps)

# Per-sample cross-entropy: -log p(y_true)
per_sample_ce = -(y_true_arr * np.log(y_prob_clipped) + (1 - y_true_arr) * np.log(1 - y_prob_clipped))

mean_ce = per_sample_ce.mean()

print(f"{'='*60}")
print(f"CROSS-ENTROPY (Dirac Delta Ground Truth)")
print(f"{'='*60}")
print(f"Mean cross-entropy: {mean_ce:.4f}")
print(f"Min  cross-entropy: {per_sample_ce.min():.4f}")
print(f"Max  cross-entropy: {per_sample_ce.max():.4f}")
print(f"Std  cross-entropy: {per_sample_ce.std():.4f}")

# ---- Per-class breakdown ----
ce_class_0 = per_sample_ce[y_true_arr == 0]
ce_class_1 = per_sample_ce[y_true_arr == 1]

print(f"\nPer-class mean cross-entropy:")
print(f"  No Convert (0): {ce_class_0.mean():.4f} (n={len(ce_class_0)})")
print(f"  Convert    (1): {ce_class_1.mean():.4f} (n={len(ce_class_1)})")

CROSS-ENTROPY (Dirac Delta Ground Truth)
Mean cross-entropy: 1.0008
Min  cross-entropy: 0.0068
Max  cross-entropy: 5.6288
Std  cross-entropy: 0.8919

Per-class mean cross-entropy:
  No Convert (0): 0.9532 (n=100)
  Convert    (1): 1.0484 (n=100)
